# GCS-backed Common Crawl Sampling and Deduplication

This notebook keeps the local Spark dedup workflow, but switches the storage edges to Google Cloud Storage. It authenticates with a service-account JSON key from `./keys`, streams sampled `.warc.wet.gz` blobs from a GCS bucket into a local JSONL working file, then uploads the final Parquet datasets to a separate GCS bucket.


## Setup and sampling parameters

This section imports the Spark helpers and GCS utilities used later, resolves the local `data/` and `keys/` paths, and defines the GCS input/output locations plus the row limits that keep the notebook laptop-friendly. Adjust these constants first when you want to trade speed for broader coverage.


In [1]:
from pathlib import Path
import shutil
import sys

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import RegexTokenizer, NGram, HashingTF, MinHashLSH

cwd = Path.cwd()
batch_dir_candidates = [cwd, cwd / 'batch']
BATCH_DIR = next((path for path in batch_dir_candidates if (path / 'gcs_storage_helpers.py').exists()), None)
if BATCH_DIR is None:
    raise FileNotFoundError('Could not locate gcs_storage_helpers.py from the current working directory.')

PROJECT_ROOT = BATCH_DIR.parent
if str(BATCH_DIR) not in sys.path:
    sys.path.append(str(BATCH_DIR))

from gcs_storage_helpers import (
    build_local_sample_from_gcs,
    build_storage_client_from_key_dir,
    list_wet_blobs,
    upload_directory_to_gcs,
)


In [2]:
DATA_DIR = BATCH_DIR / 'data'
KEY_DIR_CANDIDATES = [PROJECT_ROOT / 'keys', BATCH_DIR / 'keys']
KEYS_DIR = next((path for path in KEY_DIR_CANDIDATES if path.exists()), KEY_DIR_CANDIDATES[0])
SERVICE_ACCOUNT_KEY_FILE = 'common-crawl-deduplication-184aa125ef30.json'

GCP_PROJECT_ID = 'common-crawl-deduplication'
RAW_BUCKET_NAME = 'ccdp-raw-common-crawl-deduplication'
CRAWL_ID = 'CC-MAIN-2026-08'
RAW_OBJECT_PREFIX = f'raw/{CRAWL_ID}'
GCS_INPUT_WET_URI = f'gs://{RAW_BUCKET_NAME}/{RAW_OBJECT_PREFIX}'
GCS_OUTPUT_BASE_URI = f'gs://{RAW_BUCKET_NAME}/dedup_outputs/{CRAWL_ID}'

LOCAL_SAMPLE_PATH = DATA_DIR / 'local_wet_sample.jsonl'
LOCAL_PARQUET_STAGE_DIR = DATA_DIR / 'parquet_stage'
FINAL_PARQUET_STAGE_PATH = LOCAL_PARQUET_STAGE_DIR / 'final_docs'
DUPLICATE_AUDIT_STAGE_PATH = LOCAL_PARQUET_STAGE_DIR / 'duplicate_audit'

MAX_FILES = 10
MAX_DOCS_PER_FILE = 300
MIN_TEXT_CHARS = 200
MAX_TEXT_CHARS = 4_000
MAX_DOCS_FOR_MINHASH = 2_000
JACCARD_DISTANCE_THRESHOLD = 0.25
NGRAM_SIZE = 2
SPARK_DRIVER_MEMORY = '8g'
OVERWRITE_GCS_OUTPUT = True
KEEP_LOCAL_PARQUET_STAGE = False


In [3]:
storage_client, gcp_project_id, service_account_key_path = build_storage_client_from_key_dir(
    KEYS_DIR,
    key_filename=SERVICE_ACCOUNT_KEY_FILE,
)

wet_blobs = list_wet_blobs(storage_client, GCS_INPUT_WET_URI, max_files=MAX_FILES)

if not wet_blobs:
    raise FileNotFoundError(f'No WET gzip files found under {GCS_INPUT_WET_URI}')

print(f'GCP project: {gcp_project_id or "<unknown>"}')
print(f'Using service account key: {service_account_key_path}')
print(f'Using {len(wet_blobs)} WET files from {GCS_INPUT_WET_URI}:')
for blob in wet_blobs:
    print(' -', Path(blob.name).name)


GCP project: common-crawl-deduplication
Using service account key: /Users/leonlow/Desktop/Computer_Science/GitHub/project/common-crawl-deduplication-data-engineering-pipeline/batch/keys/common-crawl-deduplication-184aa125ef30.json
Using 10 WET files from gs://ccdp-raw-common-crawl-deduplication/raw/CC-MAIN-2026-08:
 - CC-MAIN-20260207123735-20260207153735-00833.warc.wet.gz
 - CC-MAIN-20260207214431-20260208004431-00603.warc.wet.gz
 - CC-MAIN-20260209072213-20260209102213-00672.warc.wet.gz
 - CC-MAIN-20260209194113-20260209224113-00692.warc.wet.gz
 - CC-MAIN-20260210200815-20260210230815-00103.warc.wet.gz
 - CC-MAIN-20260211052019-20260211082019-00067.warc.wet.gz
 - CC-MAIN-20260211173508-20260211203508-00229.warc.wet.gz
 - CC-MAIN-20260212085136-20260212115136-00960.warc.wet.gz
 - CC-MAIN-20260212115430-20260212145430-00304.warc.wet.gz
 - CC-MAIN-20260212145609-20260212175609-00524.warc.wet.gz


## Build a local JSONL sample

The sampled WET blobs are streamed directly from GCS and converted into a much smaller local JSONL file. That keeps the Spark part of the notebook fast and avoids forcing local Spark to parse full gzip payloads during every experiment.


In [4]:
per_file_counts, total_rows = build_local_sample_from_gcs(
    wet_blobs,
    LOCAL_SAMPLE_PATH,
    max_docs_per_file=MAX_DOCS_PER_FILE,
    min_chars=MIN_TEXT_CHARS,
    max_chars=MAX_TEXT_CHARS,
)
print(f'Wrote {total_rows} sampled documents to {LOCAL_SAMPLE_PATH}')
per_file_counts


Wrote 3000 sampled documents to /Users/leonlow/Desktop/Computer_Science/GitHub/project/common-crawl-deduplication-data-engineering-pipeline/batch/data/local_wet_sample.jsonl


{'CC-MAIN-20260207123735-20260207153735-00833.warc.wet.gz': 300,
 'CC-MAIN-20260207214431-20260208004431-00603.warc.wet.gz': 300,
 'CC-MAIN-20260209072213-20260209102213-00672.warc.wet.gz': 300,
 'CC-MAIN-20260209194113-20260209224113-00692.warc.wet.gz': 300,
 'CC-MAIN-20260210200815-20260210230815-00103.warc.wet.gz': 300,
 'CC-MAIN-20260211052019-20260211082019-00067.warc.wet.gz': 300,
 'CC-MAIN-20260211173508-20260211203508-00229.warc.wet.gz': 300,
 'CC-MAIN-20260212085136-20260212115136-00960.warc.wet.gz': 300,
 'CC-MAIN-20260212115430-20260212145430-00304.warc.wet.gz': 300,
 'CC-MAIN-20260212145609-20260212175609-00524.warc.wet.gz': 300}

In [5]:
if "spark" in globals():
    spark.stop()

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("common_crawl_deduplication_local")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/28 02:18:09 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.31 instead (on interface en0)
26/03/28 02:18:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 02:18:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load, filter, and exact-deduplicate

This section reads the JSONL sample into Spark, drops obviously unwanted rows, and creates a stronger normalized text field for exact duplicate checks. The goal is to remove clear copies before the more expensive near-duplicate stage.

In [6]:
docs_df = spark.read.json(str(LOCAL_SAMPLE_PATH))

docs_df = (
    docs_df
    .withColumn("domain", F.regexp_extract("url", r"https?://([^/]+)", 1))
    .filter(F.col("text_len") >= MIN_TEXT_CHARS)
    .filter(~F.col("url").rlike("(?i).*(porn|xxx|adult).*"))
)

print("rows:", docs_df.count())
docs_df.groupBy("source_file").count().orderBy("source_file").show(truncate=False)

rows: 2987
+-------------------------------------------------------+-----+
|source_file                                            |count|
+-------------------------------------------------------+-----+
|CC-MAIN-20260207123735-20260207153735-00833.warc.wet.gz|299  |
|CC-MAIN-20260207214431-20260208004431-00603.warc.wet.gz|300  |
|CC-MAIN-20260209072213-20260209102213-00672.warc.wet.gz|298  |
|CC-MAIN-20260209194113-20260209224113-00692.warc.wet.gz|299  |
|CC-MAIN-20260210200815-20260210230815-00103.warc.wet.gz|300  |
|CC-MAIN-20260211052019-20260211082019-00067.warc.wet.gz|298  |
|CC-MAIN-20260211173508-20260211203508-00229.warc.wet.gz|298  |
|CC-MAIN-20260212085136-20260212115136-00960.warc.wet.gz|299  |
|CC-MAIN-20260212115430-20260212145430-00304.warc.wet.gz|300  |
|CC-MAIN-20260212145609-20260212175609-00524.warc.wet.gz|296  |
+-------------------------------------------------------+-----+



In [7]:
docs_clean = (
    docs_df
    .select("source_file", "domain", "url", "text")
    # This normalization is intentionally stronger than a display-friendly clean so near-duplicate pages collide more often.
    .withColumn("text_norm", F.lower(F.col("text")))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"https?://\\S+", " "))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"www\\.\\S+", " "))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"[^a-z0-9\\s]", " "))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"\\b\\d+\\b", " "))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"\s+", " "))
    .withColumn("text_norm", F.trim("text_norm"))
    .withColumn("text_norm", F.substring("text_norm", 1, MAX_TEXT_CHARS))
    .filter(F.length("text_norm") >= MIN_TEXT_CHARS)
)

exact_duplicate_examples = (
    docs_clean
    .groupBy("text_norm")
    .agg(
        F.count("*").alias("doc_count"),
        F.collect_list(F.struct("source_file", "url")).alias("docs"),
    )
    .filter(F.col("doc_count") > 1)
    .orderBy(F.desc("doc_count"), F.asc("text_norm"))
)

before_exact = docs_clean.count()
exact_dedup = docs_clean.dropDuplicates(["text_norm"]).cache()
after_exact = exact_dedup.count()

print("before exact dedup:", before_exact)
print("after exact dedup:", after_exact)
print("exact duplicate groups:", exact_duplicate_examples.count())
exact_duplicate_examples.select("doc_count", "docs").show(10, truncate=100)

before exact dedup: 2548
after exact dedup: 2529
exact duplicate groups: 12
+---------+----------------------------------------------------------------------------------------------------+
|doc_count|                                                                                                docs|
+---------+----------------------------------------------------------------------------------------------------+
|        5|[{CC-MAIN-20260207214431-20260208004431-00603.warc.wet.gz, http://flood.umd.edu/realtime-imerg-ea...|
|        4|[{CC-MAIN-20260212115430-20260212145430-00304.warc.wet.gz, http://cargocultsoftware.com/}, {CC-MA...|
|        3|[{CC-MAIN-20260212085136-20260212115136-00960.warc.wet.gz, http://005nn.com/play/index8606-0-0.ht...|
|        3|[{CC-MAIN-20260211052019-20260211082019-00067.warc.wet.gz, http://guanhougan.net/a/449488313.html...|
|        2|[{CC-MAIN-20260211173508-20260211203508-00229.warc.wet.gz, http://a71.qq179.com/viewfile.php?id=1...|
|        2|[{CC-MAIN

## MinHash near-duplicate detection

The goal here is to compare a bounded local sample, not the full raw crawl. We keep the candidate set small so `approxSimilarityJoin` stays laptop-friendly, but use a slightly looser threshold and more forgiving text normalization so near-duplicates show up more clearly in local experiments.

In [8]:
minhash_input = (
    exact_dedup
    .orderBy("source_file", "url")
    .limit(MAX_DOCS_FOR_MINHASH)
    .withColumn("doc_id", F.monotonically_increasing_id())
)

tokenizer = RegexTokenizer(
    inputCol="text_norm",
    outputCol="tokens",
    pattern=r"\W+",
    gaps=True,
    minTokenLength=2,
)

ngram = NGram(n=NGRAM_SIZE, inputCol="tokens", outputCol="shingles")
hashing_tf = HashingTF(
    inputCol="shingles",
    outputCol="features",
    numFeatures=1 << 16,
    binary=True,
)

tokenized = tokenizer.transform(minhash_input)
shingled = ngram.transform(tokenized).filter(F.size("shingles") > 0)
features = (
    hashing_tf.transform(shingled)
    .select("doc_id", "source_file", "domain", "url", "text_norm", "features")
    .cache()
)

print("docs going into MinHash:", features.count())

docs going into MinHash: 2000


In [9]:
mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=4)
model = mh.fit(features)

pairs = (
    model.approxSimilarityJoin(features.alias("a"), features.alias("b"), JACCARD_DISTANCE_THRESHOLD, distCol="jaccard_distance")
    .select(
        F.col("datasetA.doc_id").alias("doc_id_a"),
        F.col("datasetB.doc_id").alias("doc_id_b"),
        F.col("datasetA.source_file").alias("source_file_a"),
        F.col("datasetB.source_file").alias("source_file_b"),
        F.col("datasetA.domain").alias("domain_a"),
        F.col("datasetB.domain").alias("domain_b"),
        F.col("datasetA.url").alias("url_a"),
        F.col("datasetB.url").alias("url_b"),
        F.col("jaccard_distance"),
        (F.lit(1.0) - F.col("jaccard_distance")).alias("jaccard_similarity"),
        F.substring(F.col("datasetA.text_norm"), 1, 140).alias("preview_a"),
        F.substring(F.col("datasetB.text_norm"), 1, 140).alias("preview_b"),
    )
    .filter(F.col("doc_id_a") < F.col("doc_id_b"))
    .orderBy(F.col("jaccard_distance").asc(), F.col("url_a").asc())
)

print("candidate near-duplicate pairs:", pairs.count())
pairs.show(50, truncate=80)

candidate near-duplicate pairs: 53
+--------+--------+-------------------------------------------------------+-------------------------------------------------------+------------------------------+------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------+------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|doc_id_a|doc_id_b|                                          source_file_a|                                          source_file_b|                      domain_a|                      domain_b|                                                                           url_a|                                                                           url_b|    jaccard_distance|jaccard_similarity|            

In [10]:
best_match_window = Window.partitionBy("doc_id_b").orderBy(
    F.col("jaccard_distance").asc(),
    F.col("url_a").asc(),
)

duplicate_matches = (
    pairs
    # Keep the single strongest earlier match for each removed document so the audit table stays readable.
    .withColumn("pair_rank", F.row_number().over(best_match_window))
    .filter(F.col("pair_rank") == 1)
    .cache()
)

duplicate_ids = duplicate_matches.select(F.col("doc_id_b").alias("doc_id")).distinct()
deduped_docs = minhash_input.join(duplicate_ids, on="doc_id", how="left_anti").cache()

removed_docs = (
    minhash_input.alias("removed")
    .join(duplicate_matches.alias("match"), F.col("removed.doc_id") == F.col("match.doc_id_b"))
    .select(
        F.col("match.jaccard_similarity"),
        F.col("match.source_file_a").alias("kept_source_file"),
        F.col("match.url_a").alias("kept_url"),
        F.col("removed.source_file").alias("removed_source_file"),
        F.col("removed.url").alias("removed_url"),
        F.substring(F.col("removed.text_norm"), 1, 140).alias("removed_preview"),
    )
    .orderBy(F.col("jaccard_similarity").desc(), F.col("removed_url").asc())
)

print("near-duplicate docs removed:", removed_docs.count())
removed_docs.show(20, truncate=80)
print("docs after MinHash-based filtering:", deduped_docs.count())
deduped_docs.select("source_file", "domain", "url").show(20, truncate=80)

near-duplicate docs removed: 40
+------------------+-------------------------------------------------------+--------------------------------------------------------------------------------+-------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|jaccard_similarity|                                       kept_source_file|                                                                        kept_url|                                    removed_source_file|                                                                     removed_url|                                                                 removed_preview|
+------------------+-------------------------------------------------------+--------------------------------------------------------------------------------+-------------------------------------------------------+---

In [11]:
similarity_bands = (
    pairs
    .withColumn(
        "similarity_band",
        F.when(F.col("jaccard_similarity") >= 0.95, F.lit("0.95-1.00"))
        .when(F.col("jaccard_similarity") >= 0.90, F.lit("0.90-0.95"))
        .when(F.col("jaccard_similarity") >= 0.85, F.lit("0.85-0.90"))
        .otherwise(F.lit("0.75-0.85")),
    )
    .groupBy("similarity_band")
    .agg(F.count("*").alias("pair_count"))
    .orderBy(F.desc("similarity_band"))
)

duplicate_domains = (
    removed_docs
    .withColumn("removed_domain", F.regexp_extract("removed_url", r"https?://([^/]+)", 1))
    .groupBy("removed_domain")
    .agg(F.count("*").alias("removed_docs"))
    .orderBy(F.desc("removed_docs"), F.asc("removed_domain"))
)

cross_domain_pairs = pairs.filter(F.col("domain_a") != F.col("domain_b")).count()

print("cross-domain near-duplicate pairs:", cross_domain_pairs)
similarity_bands.show(truncate=False)
duplicate_domains.show(20, truncate=False)

cross-domain near-duplicate pairs: 29


+---------------+----------+
|similarity_band|pair_count|
+---------------+----------+
|0.95-1.00      |4         |
|0.90-0.95      |11        |
|0.85-0.90      |11        |
|0.75-0.85      |27        |
+---------------+----------+

+------------------------------+------------+
|removed_domain                |removed_docs|
+------------------------------+------------+
|cimss.ssec.wisc.edu           |3           |
|fsnongji.com                  |2           |
|071zz.com                     |1           |
|080vv.com                     |1           |
|107vv.com                     |1           |
|10sss.com                     |1           |
|14400.com                     |1           |
|302888.com                    |1           |
|665dd.com                     |1           |
|973nn.com                     |1           |
|accessgroup.xtgem.com         |1           |
|aibun.jp                      |1           |
|air03-163.ppp.bekkoame.ne.jp  |1           |
|airconditioningplacerville.com

## Export final Parquet datasets

The final kept documents are first staged into local Parquet directories so local Spark can write them cheaply, then those Parquet files are uploaded to a separate GCS bucket or prefix. The kept dataset stays partitioned by `crawl_date`, and the chosen MinHash matches are uploaded to a separate audit dataset for traceability.


In [12]:
scored_features = (
    model.transform(features)
    .withColumn(
        'minhash_signature_json',
        F.to_json(F.transform('hashes', lambda x: vector_to_array(x))),
    )
    .select('doc_id', 'minhash_signature_json')
)

final_docs = (
    deduped_docs.alias('d')
    .join(docs_df.select('source_file', 'url', 'text_len').alias('raw'), on=['source_file', 'url'], how='left')
    .join(scored_features.alias('s'), on='doc_id', how='left')
    .withColumn('crawl_id', F.lit(CRAWL_ID))
    .withColumn('file_timestamp_raw', F.regexp_extract('source_file', r'CC-MAIN-(\d{14})-', 1))
    .withColumn(
        'crawl_file_timestamp',
        F.expr("try_to_timestamp(nullif(file_timestamp_raw, ''), 'yyyyMMddHHmmss')"),
    )
    .withColumn('crawl_date', F.to_date('crawl_file_timestamp'))
    .withColumn('minhash_num_tables', F.lit(4))
    .withColumn('dedup_method', F.lit('exact_plus_minhash'))
    .withColumn('record_status', F.lit('kept'))
    .select(
        'crawl_id',
        'crawl_date',
        'crawl_file_timestamp',
        'doc_id',
        'source_file',
        'domain',
        'url',
        'text',
        'text_len',
        'text_norm',
        'minhash_signature_json',
        'minhash_num_tables',
        'dedup_method',
        'record_status',
    )
)

duplicate_audit = (
    duplicate_matches
    .withColumn('crawl_id', F.lit(CRAWL_ID))
    .withColumn('match_rank', F.lit(1))
    .select(
        'crawl_id',
        'doc_id_a',
        'doc_id_b',
        'source_file_a',
        'source_file_b',
        'domain_a',
        'domain_b',
        'url_a',
        'url_b',
        'jaccard_distance',
        'jaccard_similarity',
        'match_rank',
    )
)

FINAL_PARQUET_GCS_URI = f"{GCS_OUTPUT_BASE_URI.rstrip('/')}/final_docs"
DUPLICATE_AUDIT_GCS_URI = f"{GCS_OUTPUT_BASE_URI.rstrip('/')}/duplicate_audit"

if FINAL_PARQUET_STAGE_PATH.exists():
    shutil.rmtree(FINAL_PARQUET_STAGE_PATH)
if DUPLICATE_AUDIT_STAGE_PATH.exists():
    shutil.rmtree(DUPLICATE_AUDIT_STAGE_PATH)

LOCAL_PARQUET_STAGE_DIR.mkdir(parents=True, exist_ok=True)

(
    final_docs
    .write
    .mode('overwrite')
    .partitionBy('crawl_date')
    .parquet(str(FINAL_PARQUET_STAGE_PATH))
)

(
    duplicate_audit
    .write
    .mode('overwrite')
    .parquet(str(DUPLICATE_AUDIT_STAGE_PATH))
)

final_upload_count = upload_directory_to_gcs(
    storage_client,
    FINAL_PARQUET_STAGE_PATH,
    FINAL_PARQUET_GCS_URI,
    clear_prefix=OVERWRITE_GCS_OUTPUT,
)
duplicate_upload_count = upload_directory_to_gcs(
    storage_client,
    DUPLICATE_AUDIT_STAGE_PATH,
    DUPLICATE_AUDIT_GCS_URI,
    clear_prefix=OVERWRITE_GCS_OUTPUT,
)

if not KEEP_LOCAL_PARQUET_STAGE and LOCAL_PARQUET_STAGE_DIR.exists():
    shutil.rmtree(LOCAL_PARQUET_STAGE_DIR)

print('final_docs rows:', final_docs.count())
print('uploaded final parquet files:', final_upload_count, '->', FINAL_PARQUET_GCS_URI)
print('uploaded duplicate audit files:', duplicate_upload_count, '->', DUPLICATE_AUDIT_GCS_URI)
final_docs.orderBy('crawl_date', 'source_file', 'url').show(20, truncate=80)


final_docs rows: 1960
uploaded final parquet files: 82 -> gs://ccdp-raw-common-crawl-deduplication/dedup_outputs/CC-MAIN-2026-08/final_docs
uploaded duplicate audit files: 4 -> gs://ccdp-raw-common-crawl-deduplication/dedup_outputs/CC-MAIN-2026-08/duplicate_audit
+---------------+----------+--------------------+------+-------------------------------------------------------+----------------------------+--------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------------------------------------------------------------------------------+---------------------------------------------------------+------------------+------------------+-------------+
|       crawl_id|crawl_date|crawl_file_timestamp|doc_id|                                            source_file|                      domain|                 

In [13]:
deduped_docs.count()

1960

## Notes

- Put exactly one service-account key JSON into `./keys/`, or set `SERVICE_ACCOUNT_KEY_FILE` if that folder contains multiple keys.
- The notebook still uses a local JSONL working file because that keeps the Spark portion lightweight while you experiment.
- Equal MinHash signatures are only a bucket hint, not proof of equality. Use exact text dedup for exact duplicates, and use Jaccard similarity from `approxSimilarityJoin` for near duplicates.
- If you want to scale beyond this notebook, the next step is to preprocess WET files into a structured format like JSONL or Parquet outside Spark, then run Spark on that structured dataset.
